# 类型注解 & 函数签名精讲 · 知识点

本章目标：能看懂任意第三方库（尤其是 LangChain / LangGraph）的函数签名，知道参数该怎么传。

### 1、为什么需要类型注解

In [ ]:
# Python 是动态类型语言，变量类型在运行时才确定
# 类型注解（Type Hints）是可选的，不影响运行，但有三大用处：
#
#   1. 看第三方库源码时，注解是理解「这个参数传什么」的最直接线索
#   2. IDE（PyCharm/VSCode）靠注解提供自动补全和错误提示
#   3. mypy 等静态检查工具可以在运行前发现类型错误

# 没有注解 —— 不知道每个参数该传什么
def predict(model, inputs, config):
    pass

# 有注解 —— 一眼看清楚每个参数的类型和返回值
from typing import Optional, List, Any

def predict_typed(
    model: str,
    inputs: List[str],
    config: Optional[dict] = None,
) -> List[Any]:
    pass

### 2、基础注解：变量、函数参数、返回值

In [ ]:
# 变量注解
name: str = 'Alice'
age: int = 30
score: float = 9.5
is_active: bool = True

# 只声明类型，不赋值（常见于类里的属性声明）
user_id: int

# 函数参数注解 + 返回值注解
# 格式：参数名: 类型，返回值用 -> 类型
def greet(name: str, times: int = 1) -> str:
    return f'Hello, {name}! ' * times

print(greet('Tom'))      # Hello, Tom!
print(greet('Tom', 3))   # Hello, Tom! Hello, Tom! Hello, Tom!

# 返回值是 None 时写 -> None
def log(message: str) -> None:
    print(f'[LOG] {message}')

# 注意：注解只是标记，不强制类型，运行时不会因为类型不匹配而报错
# greet(123, 'abc')  不会因为注解而报错，但 IDE 会飘红提示

### 3、Optional 与 Union

In [ ]:
from typing import Optional, Union

# Optional[X] 表示「可以是 X，也可以是 None」
# 等价于 Union[X, None]
# 常见于有默认 None 的参数，或者返回值可能找不到的情况
def find_user(user_id: int) -> Optional[str]:
    if user_id == 1:
        return 'Alice'
    return None   # 找不到时返回 None

print(find_user(1))    # Alice
print(find_user(99))   # None

# Union[X, Y] 表示「可以是 X 或者 Y」
def stringify(value: Union[str, int, float]) -> str:
    return str(value)

print(stringify(42))     # 42
print(stringify(3.14))   # 3.14
print(stringify('hi'))   # hi

# Python 3.10+ 新语法：用 | 代替 Optional / Union
# str | None     等价于   Optional[str]
# str | int      等价于   Union[str, int]
# LangChain 源码中两种写法都会出现，含义完全一样
def find_user_new(user_id: int) -> str | None:   # Python 3.10+
    return None

### 4、容器类型：list / dict / tuple / set

In [ ]:
from typing import List, Dict, Tuple, Set

# List[元素类型]
def process_names(names: List[str]) -> List[str]:
    return [name.upper() for name in names]

print(process_names(['alice', 'bob']))   # ['ALICE', 'BOB']

# Dict[键类型, 值类型]
def count_words(text: str) -> Dict[str, int]:
    result: Dict[str, int] = {}
    for word in text.split():
        result[word] = result.get(word, 0) + 1
    return result

print(count_words('a b a c a b'))   # {'a': 3, 'b': 2, 'c': 1}

# Tuple[类型1, 类型2, ...]：固定长度，每个位置类型可以不同
def get_point() -> Tuple[float, float]:
    return (1.0, 2.0)

# Tuple[str, ...]：省略号表示可变长度，所有元素同类型
def get_tags() -> Tuple[str, ...]:
    return ('python', 'async', 'langchain')

# Set[元素类型]
def unique_words(text: str) -> Set[str]:
    return set(text.split())

print(unique_words('a b a c'))   # {'a', 'b', 'c'}

# Python 3.9+ 可以直接用小写，不需要从 typing 导入
# list[str]   dict[str, int]   tuple[float, float]   set[str]
# LangChain 新版源码大量使用这种写法
def process_new(names: list[str]) -> dict[str, int]:
    return {name: len(name) for name in names}

print(process_new(['alice', 'bob']))   # {'alice': 5, 'bob': 3}

### 5、Callable / Any / Literal

In [ ]:
from typing import Callable, Any, Literal

# Callable[[参数类型列表], 返回类型]
# 表示一个可以被调用的对象（函数、lambda、实现了 __call__ 的类实例）
def apply(func: Callable[[int], str], value: int) -> str:
    return func(value)

print(apply(str, 42))               # '42'
print(apply(lambda x: f'#{x}', 5))  # '#5'

# Callable[..., 返回类型]：省略号表示参数个数/类型不限
# LangChain 里大量出现这个写法
def run_callback(callback: Callable[..., None]) -> None:
    callback()

# Any：跳过类型检查，可以接受任意类型
# 在第三方库中大量出现，表示「这个参数可以是任何东西」
def debug(value: Any) -> None:
    print(repr(value))

debug(42)
debug([1, 2, 3])
debug({'key': 'value'})

# Literal：只允许几个固定的值（类似枚举，但更轻量）
# 在 LangChain 里常见于 model_provider、mode、format 等参数
def set_log_level(level: Literal['DEBUG', 'INFO', 'WARNING', 'ERROR']) -> None:
    print(f'日志级别设为 {level}')

set_log_level('INFO')      # OK
# set_log_level('VERBOSE') # IDE 会报错：不在允许的值里

### 6、*args 和 **kwargs 的类型注解

In [ ]:
from typing import Any

# *args 的注解写的是每个元素的类型，不是整个 tuple 的类型
def sum_all(*args: int) -> int:
    return sum(args)

print(sum_all(1, 2, 3, 4))   # 10
# args 在函数内部是 tuple[int, ...]，但注解只写元素类型 int

# **kwargs 的注解写的是值的类型（键永远是 str，不需要注解）
def build_config(**kwargs: Any) -> dict:
    return dict(kwargs)

print(build_config(model='gpt-4', temperature=0.7, max_tokens=100))
# kwargs 在函数内部是 dict[str, Any]

# 常见场景：第三方库用 **kwargs: Any 透传额外参数给底层 API
def call_model(prompt: str, model: str = 'gpt-4', **kwargs: Any) -> str:
    # kwargs 里可能有 temperature、top_p、max_tokens 等
    # 库不在签名里列出这些，直接透传给底层 API
    return f'调用 {model}，附加参数：{kwargs}'

print(call_model('你好', temperature=0.5, max_tokens=200))

### 7、参数种类：位置 / 关键字 / 位置专用 / 关键字专用

In [ ]:
# Python 参数有四种传递规则，弄清楚之后看任何函数签名都不会迷惑

# ─── 普通参数：可以按位置传，也可以按关键字传 ────────────────────────
def normal(a, b):
    return a, b

print(normal(1, 2))       # 按位置
print(normal(a=1, b=2))   # 按关键字
print(normal(1, b=2))     # 混合

# ─── 位置专用参数（/ 之前的参数）：只能按位置传 ──────────────────────
# / 是分隔符，/ 前面的参数不允许用关键字方式传
def pos_only(a, b, /, c):
    return a, b, c

print(pos_only(1, 2, 3))     # OK
print(pos_only(1, 2, c=3))   # OK，c 是普通参数
# pos_only(a=1, b=2, c=3)   # 报错：a 和 b 是位置专用的

# ─── 关键字专用参数（* 之后的参数）：只能用关键字传 ──────────────────
# * 是分隔符，* 后面的参数必须用关键字方式传
def kw_only(a, *, b, c=10):
    return a, b, c

print(kw_only(1, b=2))        # OK
print(kw_only(1, b=2, c=3))   # OK
# kw_only(1, 2)               # 报错：b 是关键字专用的

# ─── 组合：同时有 / 和 * ─────────────────────────────────────────────
def mixed(pos_only, /, normal, *, kw_only):
    return pos_only, normal, kw_only

print(mixed(1, 2, kw_only=3))          # OK
print(mixed(1, normal=2, kw_only=3))   # OK

# ─── 加上 *args 和 **kwargs ─────────────────────────────────────────
def full_example(a, b=0, *args, kw1, kw2='default', **kwargs):
    print(f'a={a}, b={b}, args={args}, kw1={kw1}, kw2={kw2}, kwargs={kwargs}')

full_example(1, 2, 3, 4, kw1='必传', extra='额外')
# a=1, b=2, args=(3, 4), kw1=必传, kw2=default, kwargs={'extra': '额外'}

### 8、inspect.signature()：运行时查看函数签名

In [ ]:
import inspect
from typing import Optional, List

def chat(
    messages: List[str],
    model: str = 'gpt-4',
    temperature: float = 0.7,
    *,
    stop: Optional[List[str]] = None,
    stream: bool = False,
) -> str:
    pass

sig = inspect.signature(chat)
print('完整签名:', sig)
print()

# 遍历每个参数，查看它的种类、默认值、类型注解
for name, param in sig.parameters.items():
    kind_name = param.kind.name
    default = '无默认值' if param.default is inspect.Parameter.empty else repr(param.default)
    annotation = '无注解' if param.annotation is inspect.Parameter.empty else str(param.annotation)
    print(f'  {name}: kind={kind_name}, default={default}, annotation={annotation}')

print()
# 参数种类说明：
# POSITIONAL_OR_KEYWORD  普通参数，位置或关键字都行
# POSITIONAL_ONLY        位置专用（/ 之前）
# KEYWORD_ONLY           关键字专用（* 之后）
# VAR_POSITIONAL         *args
# VAR_KEYWORD            **kwargs

# 用 inspect 查内置函数的签名
print('len 的签名:', inspect.signature(len))
print('print 的签名:', inspect.signature(print))

### 9、实战：读懂 LangChain / LangGraph 函数签名

In [ ]:
# 下面是真实 LangChain / LangGraph 源码里的函数签名（精简版），逐行分析
from typing import Optional, List, Any, Union, Callable, Literal

# ─── 示例 1：ChatModel.invoke() ──────────────────────────────────────
#
# def invoke(
#     self,
#     input: Union[str, List[dict]],      # 可以传字符串或消息列表（二选一）
#     config: Optional[dict] = None,      # 可选配置，不传默认为 None
#     *,                                   # 之后都是关键字专用参数，必须用关键字传
#     stop: Optional[List[str]] = None,   # 停止词列表，可选
#     stream: bool = False,               # 是否流式返回
#     **kwargs: Any,                      # 额外参数透传给底层 API
# ) -> str:
#
# 怎么调：
# model.invoke('你好')                              # 最简调用
# model.invoke('你好', stop=['\n'])                  # stop 在 * 后面，必须用关键字
# model.invoke('你好', temperature=0.5)              # 额外参数走 **kwargs
print('示例 1：invoke() 签名分析完毕')

# ─── 示例 2：RunnableLambda 构造函数 ─────────────────────────────────
#
# class RunnableLambda:
#     def __init__(
#         self,
#         func: Callable[..., Any],                    # 必传：任意可调用对象
#         afunc: Optional[Callable[..., Any]] = None,  # 可选：异步版本
#     ):
#
# Callable[..., Any] = 接受任意参数、返回任意类型的可调用对象
# 怎么调：
# RunnableLambda(lambda x: x.upper())
# RunnableLambda(my_func, afunc=my_async_func)
print('示例 2：RunnableLambda 签名分析完毕')

# ─── 示例 3：LangGraph StateGraph.add_node() ─────────────────────────
#
# def add_node(
#     self,
#     node: str,                                       # 节点名称，必传，按位置传
#     action: Union[Callable[..., Any], Runnable],     # 节点执行的函数或 Runnable
#     *,                                               # 之后是关键字专用
#     metadata: Optional[dict[str, Any]] = None,
#     input: Optional[type] = None,
# ) -> None:
#
# 怎么调：
# graph.add_node('generate', my_func)
# graph.add_node('generate', chain, metadata={'version': 1})
print('示例 3：add_node() 签名分析完毕')

# ─── 读陌生函数签名的四步流程 ────────────────────────────────────────
print()
print('读陌生函数签名的通用流程：')
print('  1. 看参数名和类型注解  → 知道传什么类型的值')
print('  2. 看有没有默认值     → 知道哪些参数可以不传')
print('  3. 找 * 分隔符        → * 之后的参数必须用关键字方式传')
print('  4. 看 **kwargs        → 还有额外参数可传，查文档或源码找支持的 key')